# ruGPT-3 + LoRA V3 Tuned

In [ ]:
!pip install peft accelerate -q

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix,
)

n_threads = os.cpu_count() or 4
torch.set_num_threads(n_threads)
torch.set_num_interop_threads(min(n_threads, 4))

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | CPU threads: {torch.get_num_threads()}')

In [ ]:
DATA_PATH      = '../../data/ready_dataset.csv'
HEADLINE_COL   = 'headline_clean'
BODY_COL       = 'body_clean'
LABEL_COL      = 'label'

MODEL_NAME     = 'ai-forever/rugpt3small_based_on_gpt2'
MAX_LENGTH     = 512
BATCH_SIZE     = 4         
GRAD_ACCUM     = 16        
EPOCHS         = 15
LR             = 5e-5
WEIGHT_DECAY   = 0.02
PATIENCE       = 5
LABEL_SMOOTH   = 0.05
MAX_HOURS      = 24.0

CLASS_WEIGHTS  = [1.3, 1.0]

LORA_R         = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0.1
TARGET_MODULES = ['c_attn', 'c_proj', 'c_fc']

OUTPUT_DIR     = '../../models/llm_v3_tuned'
ADAPTER_DIR    = os.path.join(OUTPUT_DIR, 'lora_adapter')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# print(f'Model:         {MODEL_NAME}')
# print(f'max_length:    {MAX_LENGTH}')
# print(f'LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, modules={TARGET_MODULES}')
# print(f'Effective BS:  {BATCH_SIZE * GRAD_ACCUM}')
# print(f'Epochs: {EPOCHS}, patience={PATIENCE}, LR={LR}')
# print(f'Label smoothing: {LABEL_SMOOTH}')
# print(f'Class weights:   {CLASS_WEIGHTS} (fake, real)')
# print(f'Time limit: {MAX_HOURS}h')

## Данные

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[[HEADLINE_COL, BODY_COL, LABEL_COL]].dropna()
df[HEADLINE_COL] = df[HEADLINE_COL].astype(str).str.strip()
df[BODY_COL]     = df[BODY_COL].astype(str).str.strip()
df = df[(df[HEADLINE_COL] != '') & (df[BODY_COL] != '')]
df[LABEL_COL]    = pd.to_numeric(df[LABEL_COL], errors='coerce').astype(int)

train_val, test_df = train_test_split(
    df, test_size=0.1, random_state=SEED, stratify=df[LABEL_COL],
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1 / 0.9,
    random_state=SEED, stratify=train_val[LABEL_COL],
)
for d in (train_df, val_df, test_df):
    d.reset_index(drop=True, inplace=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'Balance: {dict(train_df[LABEL_COL].value_counts())}')

## Tokenization

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, headlines, bodies, labels, tokenizer, max_length):
        texts = [f"{h} | {b}" for h, b in zip(headlines, bodies)]
        self.encodings = tokenizer(
            texts, truncation=True, max_length=max_length,
            padding='max_length', return_tensors='pt',
        )
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Tokenization (max_length={MAX_LENGTH})...')
t0 = time.time()

train_dataset = NewsDataset(
    train_df[HEADLINE_COL].tolist(), train_df[BODY_COL].tolist(),
    train_df[LABEL_COL].values, tokenizer, MAX_LENGTH,
)
val_dataset = NewsDataset(
    val_df[HEADLINE_COL].tolist(), val_df[BODY_COL].tolist(),
    val_df[LABEL_COL].values, tokenizer, MAX_LENGTH,
)
test_dataset = NewsDataset(
    test_df[HEADLINE_COL].tolist(), test_df[BODY_COL].tolist(),
    test_df[LABEL_COL].values, tokenizer, MAX_LENGTH,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f'Done in {time.time()-t0:.1f}s')
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Модель

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
base_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    modules_to_save=['score'],
)

model = get_peft_model(base_model, lora_config)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Total: {total:,}, trainable: {trainable:,} ({100*trainable/total:.2f}%)')
model.print_trainable_parameters()

## Обучение

In [ ]:
opt_steps_per_epoch = len(train_loader) // GRAD_ACCUM
total_opt_steps = opt_steps_per_epoch * EPOCHS
warmup_steps = int(total_opt_steps * 0.15)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps,
)

class_weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)

print(f'Opt steps/epoch: {opt_steps_per_epoch}, total: {total_opt_steps}, warmup: {warmup_steps}')
print(f'Loss weights: fake={CLASS_WEIGHTS[0]}, real={CLASS_WEIGHTS[1]}')

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    for batch in loader:
        ids    = batch['input_ids'].to(DEVICE)
        masks  = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        outputs = model(input_ids=ids, attention_mask=masks)
        total_loss += loss_fn(outputs.logits, labels).item()
        probs = torch.softmax(outputs.logits, dim=-1).cpu()
        all_preds.extend(probs.argmax(dim=-1).numpy())
        all_labels.extend(batch['labels'].numpy())
        all_probs.extend(probs.numpy())
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    return {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'preds': preds, 'labels': labels, 'probs': np.array(all_probs),
    }

history = []
best_f1 = 0.0
no_improve = 0
t_start = time.time()

print(f'\nTraining ruGPT-3 + LoRA Tuned (r={LORA_R}, modules={TARGET_MODULES})')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    elapsed_h = (time.time() - t_start) / 3600
    if elapsed_h > MAX_HOURS:
        print(f'Time limit {MAX_HOURS}h reached, stopping.')
        break

    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, batch in enumerate(pbar):
        ids    = batch['input_ids'].to(DEVICE)
        masks  = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        loss = loss_fn(model(input_ids=ids, attention_mask=masks).logits, labels) / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f'{loss.item() * GRAD_ACCUM:.4f}')

    if len(train_loader) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_loss = epoch_loss / len(train_loader)
    val_result = evaluate(model, val_loader)

    history.append({
        'epoch': epoch,
        'train_loss': avg_loss,
        'val_loss': val_result['loss'],
        'val_acc': val_result['accuracy'],
        'val_f1': val_result['f1'],
    })

    elapsed = time.time() - t_start
    eta = elapsed / epoch * (EPOCHS - epoch)
    print(f'  Ep {epoch:2d} | loss={avg_loss:.4f} | val_loss={val_result["loss"]:.4f} | '
          f'val_acc={val_result["accuracy"]:.4f} | val_f1={val_result["f1"]:.4f} | '
          f'{elapsed/60:.0f}min (ETA {eta/60:.0f}min)')

    if val_result['f1'] > best_f1:
        best_f1 = val_result['f1']
        no_improve = 0
        model.save_pretrained(ADAPTER_DIR)
        tokenizer.save_pretrained(ADAPTER_DIR)
        print(f'  -> Saved (val_f1={best_f1:.4f})')
    else:
        no_improve += 1
        print(f'  -- no improvement ({no_improve}/{PATIENCE})')
        if no_improve >= PATIENCE:
            print(f'  -> Early stop')
            break

total_time = time.time() - t_start
print(f'\nTotal: {total_time/60:.1f} min ({total_time/3600:.1f} h)')

## Тестирование

In [ ]:
from peft import PeftModel

best_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
best_base.config.pad_token_id = tokenizer.pad_token_id
best_model = PeftModel.from_pretrained(best_base, ADAPTER_DIR)
best_model.to(DEVICE)
best_model.eval()

test_result = evaluate(best_model, test_loader)

test_metrics = {
    'accuracy':  test_result['accuracy'],
    'f1':        test_result['f1'],
    'precision': test_result['precision'],
    'recall':    test_result['recall'],
}

print('Test results:')
for k, v in test_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

print(f'\n{classification_report(test_result["labels"], test_result["preds"], target_names=["Fake", "Real"])}')

## Графики

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = [h['epoch'] for h in history]

axes[0].plot(epochs_x, [h['train_loss'] for h in history], 'o-', color='#e67e22', label='Train')
axes[0].plot(epochs_x, [h['val_loss'] for h in history], 's--', color='#8e44ad', label='Val')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()

axes[1].plot(epochs_x, [h['val_acc'] for h in history], 'o-', color='#27ae60')
axes[1].axhline(max(h['val_acc'] for h in history), color='red', ls=':', alpha=0.5)
axes[1].set(title='Val Accuracy', xlabel='Epoch', ylim=(0.5, 1.0))

axes[2].plot(epochs_x, [h['val_f1'] for h in history], 's-', color='#2980b9')
axes[2].axhline(best_f1, color='red', ls=':', alpha=0.5, label=f'Best={best_f1:.4f}')
axes[2].set(title='Val F1', xlabel='Epoch', ylim=(0.5, 1.0))
axes[2].legend()

plt.suptitle(
    f'ruGPT-3 + LoRA V3 Tuned (r={LORA_R}) -- acc={test_metrics["accuracy"]:.4f}, F1={test_metrics["f1"]:.4f}',
    fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cm = confusion_matrix(test_result['labels'], test_result['preds'])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'], ax=ax)
ax.set_title(f'V3 Tuned: ruGPT-3 + LoRA (r={LORA_R})\nAcc={test_metrics["accuracy"]:.4f}, F1={test_metrics["f1"]:.4f}',
             fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## Сохранение

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'test': test_metrics,
        'history': history,
        'training_time_min': total_time / 60,
        'config': {
            'model': MODEL_NAME,
            'max_length': MAX_LENGTH,
            'lora_r': LORA_R,
            'lora_alpha': LORA_ALPHA,
            'lora_dropout': LORA_DROPOUT,
            'target_modules': TARGET_MODULES,
            'effective_batch': BATCH_SIZE * GRAD_ACCUM,
            'epochs': EPOCHS,
            'lr': LR,
            'label_smoothing': LABEL_SMOOTH,
            'class_weights': CLASS_WEIGHTS,
        },
    }, f, indent=4, ensure_ascii=False)

pred_df = test_df[[HEADLINE_COL, BODY_COL, LABEL_COL]].copy()
pred_df['pred']      = test_result['preds']
pred_df['prob_real'] = test_result['probs'][:, 1]
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False)

errors = test_result['preds'] != test_result['labels']
print(f'Correct: {(~errors).sum()}/{len(test_result["labels"])}')
print(f'FP: {((test_result["labels"]==0) & (test_result["preds"]==1)).sum()}')
print(f'FN: {((test_result["labels"]==1) & (test_result["preds"]==0)).sum()}')
print(f'Training time: {total_time/3600:.1f}h')